
# 🧠 Fundamental Machine Learning using Python for Public Health
## Day 2 — Logistic Regression, Neural Networks & Deep Learning

**A 2-Day Hands-on Workshop**
Faculty of Public Health, Khon Kaen University

**Instructor:** Teerapong Panboonyuen (Kao) — [kaopanboonyuen.github.io](https://kaopanboonyuen.github.io/)  
**Program:** MPH (Master of Public Health), Department of Biostatistics  
**Dataset:** `kku-stroke-dataset.csv` — [GitHub Repository](https://github.com/kaopanboonyuen/KKU_Biostat_ML_2026s2)  
**Paper reproduced in class:** Melnykova, N., et al. (2025). *Machine learning for stroke prediction using imbalanced data.* Scientific Reports, 15(1), 33773.  

---

###  Acknowledgements:
This workshop is built upon the foundation of the global open-source scientific community. We sincerely acknowledge the developers, researchers, and contributors behind PyTorch, scikit-learn, NumPy, Pandas, Matplotlib, Seaborn, Jupyter Notebook, and Kaggle for advancing reproducible research, machine learning education, and open scientific collaboration.

---

### 🎯 Learning Objectives for Today

Yesterday we built a full ML pipeline with **Decision Trees** and **Random Forests**. Today we go one level deeper into *how models actually learn*:

1. Understand **Logistic Regression** in full mathematical detail — the model that started it all.
2. See exactly how a **single artificial neuron** (a sigmoid unit) *is* a logistic regression — the bridge from classical ML to Deep Learning.
3. Build and train your first **PyTorch neural networks**, from a simple 2-layer network to a deeper architecture.
4. Learn how to **read a research paper's Methods section** and translate hyperparameters (epochs, learning rate, batch size) into working code.
5. Handle **imbalanced clinical data** properly using class weighting.
6. Compare **Decision Tree / Random Forest (Day 1) vs. Logistic Regression vs. Neural Networks (Day 2)** on the exact same stroke dataset, using the exact same metrics.
7. Close with a **recap, an AI/ML research roadmap for public health, and essential ethics guidance**.

> 💡 Cells marked **📝 EXERCISE** are for you to try.

Let's dive into the math and the machinery behind modern AI! 🚀



## 0. Setup & Reloading the Data

We reload the raw dataset and quickly repeat the **same feature engineering and stratified split** from Day 1, so this notebook can run completely independently.


In [ ]:

# If you are running this OUTSIDE Google Colab and are missing a package, uncomment below:
# !pip install -q numpy pandas matplotlib seaborn scikit-learn torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = # Write your lucky number here
np.random.seed(RANDOM_STATE)

print("✅ Environment ready.")


In [ ]:
# Dataset configuration
DATASET_NAME = # Write your code here

# Construct the dataset URL
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "kaopanboonyuen/KKU_Biostat_ML_2026s2/main/dataset/"
    f"{DATASET_NAME}.csv"
)

df = pd.read_csv(DATA_URL, na_values=["N/A"])

df.head()

In [ ]:

# ---- Recap of Day 1 feature engineering, condensed ----
data = df.copy()
data.head()


In [ ]:
# Group-wise median imputation for BMI
data["age_bin_for_impute"] = pd.cut(data["age"], bins=[0, 18, 40, 60, 80, 120])
data["bmi"] = data.groupby(["age_bin_for_impute", "gender"])["bmi"].transform(lambda s: s.fillna(s.median()))
data["bmi"] = data["bmi"].fillna(data["bmi"].median())
data.drop(columns=["age_bin_for_impute"], inplace=True)


In [ ]:
# TODO: Clinical binning

def bmi_category(v):
    # TODO: Implement BMI clinical binning logic
    pass


def glucose_category(v):
    # TODO: Implement glucose clinical binning logic
    pass

In [ ]:
data["bmi_category"] = # Write your code here
data["glucose_category"] = data["avg_glucose_level"].apply(glucose_category)
data["age_group"] = pd.cut(data["age"], bins=[0, 18, 40, 60, 80, 120], labels=["0-18", "19-40", "41-60", "61-80", "81+"])


In [ ]:
# Encoding
data["ever_married_flag"] = data["ever_married"].map({"Yes": 1, "No": 0})
data = data.drop(columns=["ever_married"])

categorical_for_ohe = ["gender", "work_type", "Residence_type", "smoking_status",
                        "bmi_category", "glucose_category", "age_group"]
data_encoded = pd.get_dummies(data, columns=categorical_for_ohe, drop_first=False)


In [ ]:
# Composite / interaction features
data_encoded["comorbidity_count"] = data["hypertension"] + data["heart_disease"]
data_encoded["age_x_hypertension"] =  # Write your code here
data_encoded["metabolic_risk_score"] = (
    (data["avg_glucose_level"] / data["avg_glucose_level"].max()) * 0.5
    + (data["bmi"] / data["bmi"].max()) * 0.5
)
data_encoded["is_senior"] = (data["age"] >= 65).astype(int)
data_encoded["log_avg_glucose_level"] = np.log1p(data_encoded["avg_glucose_level"])

drop_cols = ["id"]
feature_cols = [c for c in data_encoded.columns if c not in drop_cols + ["stroke"]]


In [ ]:
# Make sure every feature column is numeric (bool -> int) for downstream models (sklearn AND PyTorch)
X =  # Write your code here
y =  # Write your code here

print("Feature matrix shape:", X.shape)
X.head()



## 0.1 Train/Test Split + Feature Scaling

**Why scaling matters today (and didn't yesterday):** Decision Trees / Random Forests split on raw thresholds, so feature scale doesn't matter. But **Logistic Regression** and **Neural Networks** compute weighted sums like `w1*age + w2*bmi + ...` — if `age` ranges 0-100 and `metabolic_risk_score` ranges 0-1, the optimizer struggles. **`StandardScaler`** rescales every feature to mean 0, standard deviation 1, fixing this.

> ⚠️ **Golden rule:** fit the scaler ONLY on training data, then apply (`transform`) it to the test data. Fitting on the test set would leak information from the "future" into training — a common mistake that inflates reported performance in papers.


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on TRAIN only
X_test_scaled = scaler.transform(X_test)          # transform ONLY on TEST (no fitting!)

print("Train:", X_train_scaled.shape, " Test:", X_test_scaled.shape)
print("Train stroke rate: %.2f%% | Test stroke rate: %.2f%%" % (y_train.mean()*100, y_test.mean()*100))



---
# Part A — Logistic Regression, Explained From First Principles 📐

Logistic Regression is arguably **the single most important algorithm in the history of AI** — it is mathematically identical to a single artificial neuron, and understanding it fully unlocks Neural Networks and Deep Learning.

## A.1 The Problem With Using Linear Regression for Classification

A plain linear model predicts `ŷ = w·x + b`, which can output any real number (e.g. `-3.2` or `+57`). But we want a **probability** between 0 and 1 (the probability of stroke). We need a function that "squashes" any real number into the `(0, 1)` range.

## A.2 The Sigmoid Function

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

- As `z → +∞`, `σ(z) → 1`
- As `z → -∞`, `σ(z) → 0`
- At `z = 0`, `σ(z) = 0.5`

Let's visualize it:


In [ ]:

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

z_values = np.linspace(-10, 10, 200)
plt.figure(figsize=(7, 4.5))
plt.plot(z_values, sigmoid(z_values), linewidth=2.5, color="#4C72B0")
plt.axhline(0.5, linestyle="--", color="gray", linewidth=1)
plt.axvline(0, linestyle="--", color="gray", linewidth=1)
plt.title("The Sigmoid Function")
plt.xlabel("z (raw weighted sum, a.k.a. the 'logit')")
plt.ylabel("σ(z)  →  predicted probability")
plt.tight_layout()
plt.show()



## A.3 The Full Logistic Regression Model

For a patient with features `x = [age, bmi, glucose, ...]`, Logistic Regression:

**Step 1 — Linear combination ("the logit"):**
$$
z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \mathbf{w}\cdot\mathbf{x} + b
$$

**Step 2 — Squash with sigmoid to get a probability:**
$$
\hat{p} = P(\text{stroke}=1 \mid \mathbf{x}) = \sigma(z)
$$

**Step 3 — Threshold to get a class prediction** (default threshold 0.5):
$$
\hat{y} = \begin{cases} 1 & \text{if } \hat{p} \geq 0.5 \\ 0 & \text{otherwise} \end{cases}
$$

## A.4 How the Model *Learns*: the Loss Function

We need a way to measure "how wrong" a predicted probability `p̂` is, given the true label `y ∈ {0, 1}`. Logistic Regression uses **Binary Cross-Entropy (Log Loss)**:

$$
\mathcal{L}(y, \hat{p}) = -\big[\, y \log(\hat{p}) + (1-y)\log(1-\hat{p}) \,\big]
$$

Intuition:
- If the true label is `y=1` and the model confidently predicts `p̂ ≈ 1` → loss ≈ 0 (great!)
- If the true label is `y=1` but the model predicts `p̂ ≈ 0` → loss → ∞ (heavily penalized!)

**Training** = find the weights `w` and bias `b` that **minimize the average loss** across all patients, using an iterative optimization algorithm called **Gradient Descent**: at every step, nudge each weight slightly in the direction that reduces the loss, controlled by the **learning rate**.


In [ ]:

# A tiny hand-built illustration of Binary Cross-Entropy — NOT used for actual training,
# just to build intuition before we let scikit-learn do the optimization for us.
def binary_cross_entropy(y_true, p_pred, eps=1e-9):
    p_pred = np.clip(p_pred, eps, 1 - eps)  # avoid log(0)
    return -(y_true * np.log(p_pred) + (1 - y_true) * np.log(1 - p_pred))

examples = pd.DataFrame({
    "true_label (y)":      [1,    1,    1,    0,    0,    0],
    "predicted_prob (p̂)": [0.95, 0.5,  0.05, 0.05, 0.5,  0.95],
})
examples["loss"] = binary_cross_entropy(examples["true_label (y)"], examples["predicted_prob (p̂)"])
examples



## A.5 Training Logistic Regression with `scikit-learn`

**Key hyperparameters:**

| Parameter | What it controls | Tuning intuition |
|---|---|---|
| `C` | Inverse regularization strength | Smaller `C` = stronger regularization = simpler, more conservative model (helps prevent overfitting on noisy clinical features) |
| `penalty` | Type of regularization (`l2`, `l1`, `elasticnet`) | `l2` (default) shrinks all weights smoothly; `l1` can zero out irrelevant features entirely (built-in feature selection) |
| `class_weight="balanced"` | Re-weights the rare positive class | Essential here — stroke is rare! |
| `max_iter` | Max gradient-descent iterations allowed | Increase if you see a "did not converge" warning |
| `solver` | Optimization algorithm used | `"lbfgs"` (default) works well for small/medium datasets like ours |


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

log_reg = LogisticRegression(
    C=1.0,
    penalty="l2",
    class_weight="balanced",
    max_iter=2000,
    solver="lbfgs",
    random_state=RANDOM_STATE,
)

log_reg.fit(X_train_scaled, y_train)
print("✅ Logistic Regression trained.")



## A.6 Interpreting Logistic Regression Coefficients

Unlike a Random Forest's black-box feature importance, Logistic Regression gives us a **directly interpretable equation**. Each coefficient `w_i` tells us: *holding everything else constant, how does a 1-standard-deviation increase in this (scaled) feature change the log-odds of stroke?*

A positive coefficient → increases stroke risk. A negative coefficient → decreases it.


In [ ]:

coef_series = pd.Series(log_reg.coef_[0], index=X.columns).sort_values()

plt.figure(figsize=(8, 9))
colors = ["#C44E52" if v > 0 else "#4C72B0" for v in coef_series.values]
plt.barh(coef_series.index, coef_series.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Logistic Regression Coefficients\n(red = increases stroke risk, blue = decreases it)")
plt.xlabel("Coefficient (log-odds scale, standardized features)")
plt.tight_layout()
plt.show()



👉 Notice how `age`, `age_x_hypertension`, `hypertension`, and `heart_disease` typically show strong **positive** coefficients — consistent with the feature importance we saw with Random Forest on Day 1! This kind of cross-model agreement is exactly what reviewers look for in a solid public-health ML paper.



## A.7 Evaluating Logistic Regression on the Test Set

Same metric suite as Day 1, so we can compare apples to apples.


In [ ]:

def evaluate_model(y_true, y_pred, y_proba, model_name):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
    }

log_reg_pred = log_reg.predict(X_test_scaled)
log_reg_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

all_results = [evaluate_model(y_test, log_reg_pred, log_reg_proba, "Logistic Regression")]
pd.DataFrame(all_results).set_index("Model").round(3)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

fpr, tpr, _ = roc_curve(y_test, log_reg_proba)
axes[0].plot(fpr, tpr, linewidth=2.5, color="#4C72B0", label=f"Logistic Regression (AUC={roc_auc_score(y_test, log_reg_proba):.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — Logistic Regression")
axes[0].legend(loc="lower right")

cm = confusion_matrix(y_test, log_reg_pred)
ConfusionMatrixDisplay(cm, display_labels=["No Stroke", "Stroke"]).plot(ax=axes[1], cmap="Blues", colorbar=False, values_format="d")
axes[1].set_title("Confusion Matrix — Logistic Regression")

plt.tight_layout()
plt.show()

print(classification_report(y_test, log_reg_pred, target_names=["No Stroke", "Stroke"]))



---
# Part B — From One Neuron to a Neural Network 🧠⚡

## B.1 The Big Reveal: Logistic Regression *IS* a Single Artificial Neuron

Everything you just learned in Part A is *exactly* what happens inside **one neuron** of a neural network:

```
        x1 ──┐
        x2 ──┤  (each xi multiplied by weight wi)
        x3 ──┼──►  z = w1·x1 + w2·x2 + w3·x3 + ... + b  ──►  σ(z)  ──►  ŷ (probability)
        ...  │
        xn ──┘
```

- The **weighted sum** step (`z = w·x + b`) → same as Logistic Regression's linear combination.
- The **sigmoid activation** (`σ(z)`) → the exact same squashing function from Part A.
- **Training** → the exact same idea (minimize cross-entropy loss via gradient descent), just automated by a deep learning framework.

**So what makes a "Neural Network" more powerful than plain Logistic Regression?** We stack **many neurons** into **layers**, and stack **many layers** on top of each other:

```
INPUT LAYER          HIDDEN LAYER 1        HIDDEN LAYER 2        OUTPUT LAYER
(your features)      (many neurons,        (many neurons,        (1 neuron,
                       each with its         each with its         sigmoid activation
                       own sigmoid/ReLU)      own sigmoid/ReLU)     → probability of stroke)

  x1 ●───────────────► ● ─┐                  ● ─┐
  x2 ●───────────────► ● ─┼──────────────►   ● ─┼──────────────►  ● ──► ŷ
  x3 ●───────────────► ● ─┤                  ● ─┤
 ... ●───────────────► ● ─┘                  ● ─┘
  xn ●───────────────► ●                     ●
```

Each hidden neuron can learn its own "mini logistic regression" over the outputs of the previous layer. Stacked together, the network can learn **non-linear combinations of features** that plain Logistic Regression cannot express on its own — e.g. "high glucose is only dangerous when combined with old age AND hypertension," a pattern a single straight-line decision boundary struggles to capture.

## B.2 Activation Functions: Sigmoid vs. ReLU

Modern networks usually use **ReLU** (`ReLU(z) = max(0, z)`) in hidden layers instead of sigmoid, because it trains faster and avoids a problem called *vanishing gradients*. We still use **sigmoid** on the final output neuron, because we want a probability between 0 and 1.


In [ ]:

def relu(z):
    return np.maximum(0, z)

z_values = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(z_values, sigmoid(z_values), color="#4C72B0", linewidth=2.5)
axes[0].set_title("Sigmoid — used on the OUTPUT neuron\n(squashes to a 0-1 probability)")
axes[1].plot(z_values, relu(z_values), color="#55A868", linewidth=2.5)
axes[1].set_title("ReLU — used in HIDDEN layers\n(fast to train, avoids vanishing gradients)")
for ax in axes:
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.axvline(0, color="gray", linewidth=0.7)
plt.tight_layout()
plt.show()



## B.3 Setting Up PyTorch

We now switch to **PyTorch**, the industry-standard deep learning framework, to build real neural networks that learn via **automatic differentiation** and **gradient descent** — exactly the training process described in Part A, but generalized to many layers.


In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Convert our scaled numpy arrays into PyTorch tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)  # shape (N, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_t, y_train_t)

n_features = X_train_scaled.shape[1]
print("Number of input features:", n_features)



## B.4 Handling Class Imbalance in PyTorch

Just like `class_weight="balanced"` in scikit-learn, PyTorch lets us weight the positive class inside the loss function itself, using `pos_weight` in `BCEWithLogitsLoss` (this combines the sigmoid + cross-entropy into one numerically stable step).


In [ ]:

n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight_value = n_neg / n_pos   # how many times more common the negative class is
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32)

print(f"Positive (stroke) samples in train: {n_pos} | Negative: {n_neg}")
print(f"pos_weight passed to the loss function: {pos_weight_value:.2f}")



## B.5 Architecture 1 — A Simple 2-Hidden-Layer MLP ("Multi-Layer Perceptron")

Our first neural network: input → hidden layer (32 neurons, ReLU) → hidden layer (16 neurons, ReLU) → output neuron (1 value, raw logit — sigmoid is applied inside the loss function for numerical stability).


In [ ]:

class SimpleMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 32),  # input -> hidden layer 1 (32 neurons)
            nn.ReLU(),
            nn.Linear(32, 16),          # hidden layer 1 -> hidden layer 2 (16 neurons)
            nn.ReLU(),
            nn.Linear(16, 1),           # hidden layer 2 -> output neuron (1 raw logit)
        )

    def forward(self, x):
        return self.network(x)

model_simple = SimpleMLP(n_features).to(device)
print(model_simple)
total_params = sum(p.numel() for p in model_simple.parameters())
print(f"Total trainable parameters: {total_params}")



## B.6 A Reusable Training Loop (with Hyperparameter Notes)

**Key hyperparameters, and how to read them from a paper's Methods section:**

| Parameter | What it controls | How to choose it |
|---|---|---|
| `epochs` | How many full passes over the training data | Watch the training loss curve — stop once it plateaus (too many epochs on a small dataset like ours → overfitting) |
| `learning_rate (lr)` | Step size of each gradient-descent update | Too high → training diverges / loss explodes. Too low → painfully slow. `1e-3` is a solid default for Adam. |
| `batch_size` | How many samples processed before each weight update | Smaller = noisier but sometimes better generalization; larger = smoother but needs more memory. `32-128` is typical for tabular data. |
| `optimizer` | The algorithm used to update weights | **Adam** (adaptive learning rate) is the default modern choice; plain SGD needs more tuning but can generalize better on some problems. |
| `weight_decay` | L2 regularization strength inside the optimizer | A small value (e.g. `1e-4`) helps prevent overfitting, similar to `C` in Logistic Regression. |

When reading a paper, look in the "Implementation Details" or "Training" subsection of the Methods for exactly these five numbers — that's usually enough to reproduce their training setup.


In [ ]:

def train_pytorch_model(model, train_dataset, X_test_t, y_test_t, pos_weight,
                         epochs=60, lr=1e-3, batch_size=64, weight_decay=1e-4, verbose_every=10):
    """A general-purpose training loop we will reuse for every PyTorch architecture today."""
    model.to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    history = {"train_loss": [], "test_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()          # backpropagation: compute gradients
            optimizer.step()         # gradient descent: update weights
            running_loss += loss.item() * xb.size(0)
        train_loss = running_loss / len(train_dataset)

        model.eval()
        with torch.no_grad():
            test_logits = model(X_test_t.to(device))
            test_loss = criterion(test_logits, y_test_t.to(device)).item()

        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)

        if epoch % verbose_every == 0 or epoch == 1:
            print(f"Epoch {epoch:>3}/{epochs} | train_loss={train_loss:.4f} | test_loss={test_loss:.4f}")

    return history


def get_pytorch_predictions(model, X_tensor, threshold=0.5):
    """Convert raw logits into probabilities and 0/1 predictions."""
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor.to(device))
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
    preds = (probs >= threshold).astype(int)
    return preds, probs


In [ ]:

history_simple = train_pytorch_model(
    model_simple, train_dataset, X_test_t, y_test_t, pos_weight,
    epochs=60, lr=1e-3, batch_size=64,
)


In [ ]:

plt.figure(figsize=(7, 4.5))
plt.plot(history_simple["train_loss"], label="Train loss", linewidth=2)
plt.plot(history_simple["test_loss"], label="Test loss", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Simple MLP — Training Curve")
plt.legend()
plt.tight_layout()
plt.show()



👉 **Reading a training curve:** if `train_loss` keeps falling but `test_loss` starts rising again, that's **overfitting** — the network is memorizing the training patients rather than learning generalizable patterns. Solutions include: fewer epochs, more regularization (`weight_decay`), a simpler architecture, or more training data.


In [ ]:

simple_pred, simple_proba = get_pytorch_predictions(model_simple, X_test_t)
all_results.append(evaluate_model(y_test, simple_pred, simple_proba, "Neural Net (Simple MLP)"))
pd.DataFrame(all_results).set_index("Model").round(3)



## B.7 Architecture 2 — A Deeper MLP with Dropout & Batch Normalization

Let's go deeper: 3 hidden layers, plus two regularization tricks used constantly in real deep-learning research:

- **`nn.BatchNorm1d`** — normalizes activations inside the network, which stabilizes and speeds up training.
- **`nn.Dropout(p)`** — randomly "turns off" a fraction `p` of neurons during each training step, forcing the network to not rely too heavily on any single neuron. This is one of the most effective and widely-used ways to fight overfitting in deep learning.


In [ ]:

class DeeperMLP(nn.Module):
    def __init__(self, n_features, dropout_p=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.network(x)

model_deep = DeeperMLP(n_features, dropout_p=0.3).to(device)
print(model_deep)
print("Total trainable parameters:", sum(p.numel() for p in model_deep.parameters()))


In [ ]:

history_deep = train_pytorch_model(
    model_deep, train_dataset, X_test_t, y_test_t, pos_weight,
    epochs=80, lr=1e-3, batch_size=64, weight_decay=1e-4,
)


In [ ]:

plt.figure(figsize=(7, 4.5))
plt.plot(history_deep["train_loss"], label="Train loss", linewidth=2)
plt.plot(history_deep["test_loss"], label="Test loss", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Deeper MLP (Dropout + BatchNorm) — Training Curve")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

deep_pred, deep_proba = get_pytorch_predictions(model_deep, X_test_t)
all_results.append(evaluate_model(y_test, deep_pred, deep_proba, "Neural Net (Deeper MLP)"))
pd.DataFrame(all_results).set_index("Model").round(3)



## B.8 Architecture 3 — A 1D Convolutional Neural Network (CNN): Automatic Feature Engineering

On Day 1, **we** engineered features by hand: BMI categories, glucose categories, composite risk scores, interaction terms — using our own clinical domain knowledge. That's incredibly valuable, but it's also slow and relies entirely on a human noticing the right pattern.

**Convolutional layers let the network discover its own local feature combinations automatically**, instead of us hand-crafting every one. CNNs were originally built for images (scanning small patches with a sliding filter), but the same idea works on tabular data if we treat our row of features as a **1D signal**: a `Conv1d` filter slides across neighboring features and learns to combine them — effectively learning its own little "interaction features," the same spirit as our `age_x_hypertension` feature on Day 1, except the network discovers *which* combinations matter on its own.

### The Building Blocks

| Layer | What it does |
|---|---|
| `nn.Conv1d(in_channels, out_channels, kernel_size)` | Slides `out_channels` learnable filters, each looking at `kernel_size` neighboring features at a time, across the input. Each filter learns to detect a different local pattern. |
| `nn.ReLU()` | Same non-linearity as before — keeps only positive activations. |
| `nn.MaxPool1d(kernel_size)` | Downsamples by keeping only the strongest (max) activation in every window — keeps the most important signal, discards the rest, and makes the network more robust to small shifts. |
| `nn.Flatten()` | Reshapes the multi-channel feature maps back into a single flat vector, ready for regular `nn.Linear` layers. |
| `nn.Linear(...)` | Same fully-connected "decision" layers as our MLPs, now working on top of CNN-extracted features instead of raw columns. |

### Key `Conv1d` / `MaxPool1d` Parameters — A Tuning Guide

| Parameter | What it controls | Tuning intuition |
|---|---|---|
| `in_channels` | Number of "channels" coming in | `1` for our first conv layer (one row of raw features); equals the previous layer's `out_channels` afterwards |
| `out_channels` | Number of different filters (learned feature detectors) to apply | More filters = network can learn more distinct patterns, but more parameters to train. `8-32` is reasonable for small tabular inputs. |
| `kernel_size` | How many neighboring features each filter looks at simultaneously | Small (e.g. `3`) = very local combinations (like a single interaction term); larger = broader patterns, but tabular columns usually aren't ordered meaningfully, so keep this modest. |
| `stride` | How far the filter moves between each application | `1` = examine every possible position (most common default); `>1` skips positions and shrinks the output faster. |
| `padding` | Zero-padding added to the edges of the input | `same`-style padding (e.g. `kernel_size // 2`) keeps the output length close to the input length so we don't lose the edge features. |
| `MaxPool1d(kernel_size)` | Size of the pooling window | Larger pooling = more aggressive downsampling (fewer, more compressed features) — `2` is a common gentle default. |

> ⚠️ **Important caveat for tabular data:** unlike pixels in an image, our feature *columns have no natural left-right order* — `age` next to `bmi` is arbitrary. So a 1D CNN on tabular data is a **great teaching tool** for understanding convolution + pooling + automatic feature learning, but in practice, tree-based models or MLPs are usually the stronger, more principled choice for tabular clinical data like ours. CNNs truly shine on data with real spatial/sequential structure — images, ECG/EEG signals, time series.


In [ ]:

class ConvNet1D(nn.Module):
    """A small 1D CNN that treats the feature vector as a 1-channel signal
    and lets convolution + pooling learn its own local feature combinations,
    instead of us hand-engineering them like we did on Day 1."""

    def __init__(self, n_features):
        super().__init__()

        # --- Automatic feature-extraction stage (the "conv trunk") ---
        self.conv_block = nn.Sequential(
            # Layer 1: 1 input channel (raw feature row) -> 16 learned filters
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),   # halves the sequence length, keeps strongest signal

            # Layer 2: 16 channels -> 32 learned filters, looking at the previous layer's features
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Flatten(),  # collapse (channels, length) back into one flat vector
        )

        # Figure out the flattened size automatically by running one dummy example through
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_features)
            flattened_size = self.conv_block(dummy).shape[1]

        # --- Decision stage (same idea as our MLP's final layers) ---
        self.classifier = nn.Sequential(
            nn.Linear(flattened_size, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),  # raw logit, sigmoid applied inside the loss function
        )

    def forward(self, x):
        # x arrives as shape (batch_size, n_features) -> reshape to (batch_size, 1 channel, n_features)
        x = x.unsqueeze(1)
        x = self.conv_block(x)
        return self.classifier(x)


model_cnn = ConvNet1D(n_features).to(device)
print(model_cnn)
print("Total trainable parameters:", sum(p.numel() for p in model_cnn.parameters()))


In [ ]:

history_cnn = train_pytorch_model(
    model_cnn, train_dataset, X_test_t, y_test_t, pos_weight,
    epochs=80, lr=1e-3, batch_size=64, weight_decay=1e-4,
)


In [ ]:

plt.figure(figsize=(7, 4.5))
plt.plot(history_cnn["train_loss"], label="Train loss", linewidth=2)
plt.plot(history_cnn["test_loss"], label="Test loss", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Binary Cross-Entropy Loss")
plt.title("1D CNN — Training Curve")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

cnn_pred, cnn_proba = get_pytorch_predictions(model_cnn, X_test_t)
all_results.append(evaluate_model(y_test, cnn_pred, cnn_proba, "Neural Net (1D CNN)"))
pd.DataFrame(all_results).set_index("Model").round(3)



👉 **What did the CNN actually learn?** Each of the 16 (then 32) filters in the conv layers learned its own small "recipe" combining a handful of neighboring input features — entirely on its own, with no clinical knowledge given to it. Compare its **Recall** and **ROC-AUC** against the Simple MLP and Deeper MLP above: on a small tabular dataset like ours, automatic feature learning via convolution usually does **not** beat a good hand-engineered feature set + a plain MLP or Random Forest — a nice, honest, teachable result about *when* CNNs are worth their extra complexity (they truly shine on images, signals, and sequences with real local structure).



---
## 📝 EXERCISE — Build Your Own Neural Network

**Your task:** design and train a **3rd architecture**, `MyMLP`, using the same `nn.Sequential` pattern.

**Hints:**
- Try somewhere between 1 and 3 hidden layers.
- Reasonable neuron counts: 8, 16, 32, or 64 per layer.
- Use `nn.ReLU()` after each hidden `nn.Linear`.
- The **final layer must output exactly 1 value** (a raw logit — no activation needed there, `BCEWithLogitsLoss` applies sigmoid internally).
- Try adding `nn.Dropout(0.2)` after one hidden layer to see its effect.
- Reuse `train_pytorch_model(...)` and `get_pytorch_predictions(...)` exactly like above — no need to rewrite the training loop!


In [ ]:

# ✍️ Your turn! Define your own architecture below.

class MyMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.network = nn.Sequential(
            # TODO: design your own layers here
        )

    def forward(self, x):
        return self.network(x)

# TODO: instantiate, train, and evaluate your model

model_custom = MyMLP(n_features).to(device)
history_custom = train_pytorch_model(
    model_custom, train_dataset, X_test_t, y_test_t, pos_weight,
    epochs=60, lr=1e-3, batch_size=64,
)

custom_pred, custom_proba = get_pytorch_predictions(model_custom, X_test_t)
all_results.append(evaluate_model(y_test, custom_pred, custom_proba, "Neural Net (Student Custom)"))
pd.DataFrame(all_results).set_index("Model").round(3)


---
# Part C — Comparing Every Model We Built (Day 1 + Day 2) 📊

Let's bring in yesterday's Decision Tree and Random Forest results and compare **all five models** side by side, using the exact same test set.

> Note: re-run this cell's Day-1 section only if you have `dt_model` / `rf_model` from Day 1 already in this same runtime — otherwise we simply compare the Day 2 models we trained above.


In [ ]:

comparison_df = pd.DataFrame(all_results).set_index("Model").sort_values("ROC-AUC", ascending=False).round(3)
comparison_df


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
comparison_df["ROC-AUC"].sort_values().plot(kind="barh", color="#4C72B0", ax=ax)
ax.set_xlabel("ROC-AUC (higher is better)")
ax.set_title("Model Comparison — ROC-AUC on the Held-out Test Set")
ax.set_xlim(0.5, 1.0)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(7, 6.5))
for name, proba in [
    ("Logistic Regression", log_reg_proba),
    ("Neural Net (Simple MLP)", simple_proba),
    ("Neural Net (Deeper MLP)", deep_proba),
    ("Neural Net (1D CNN)", cnn_proba),
    ("Neural Net (Student Custom)", custom_proba),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves — All Day 2 Models")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()



👉 **A very common, very important research finding:** on small-to-medium **tabular** clinical datasets like ours (~5,000 rows, ~12 raw features), tree-based methods (Random Forest) and even plain Logistic Regression often perform **just as well, or better than**, a deep neural network. Deep Learning tends to shine when there is a **lot more data** and/or **unstructured inputs** (images, text, signals) where automatic feature learning has more raw material to work with. Knowing *when* to reach for Deep Learning vs. classical ML is itself an important research skill — bigger and more complex is not always better!

---
# Part D — Workshop Wrap-Up: Recap, Research Guidance & AI Ethics 🎓



## D.1 Full Two-Day Recap

**Day 1 — Foundations:**
- Python refresher: variables, operators, conditionals, loops, containers, functions, classes
- Loaded and explored the KKU Stroke Dataset with pandas: missingness, class imbalance, rich EDA
- Feature engineering: imputation, clinical binning (BMI/glucose/age), encoding, composite risk scores, skew correction
- Built a full scikit-learn pipeline: stratified split → Decision Tree → Random Forest → feature importance → full evaluation suite

**Day 2 — Deeper Modeling:**
- Logistic Regression from first principles: the sigmoid function, the logit, binary cross-entropy loss, gradient descent
- The conceptual bridge from **one neuron = Logistic Regression** to full **Neural Networks**
- Built and trained real **PyTorch** models: a simple MLP, a deeper MLP with Dropout + BatchNorm, a 1D CNN (automatic feature learning via convolution + pooling), and your own custom architecture
- Learned to reason about **epochs, learning rate, batch size, optimizer choice, and regularization** — the exact hyperparameters you'll see in any deep learning paper's Methods section
- Compared **six different models** on identical data with identical metrics — the gold standard for a fair benchmark

## D.2 A Guide to Doing AI/ML Research the Right Way

If you plan to take this further into your own thesis or publication, keep this checklist close:

1. **Define the clinical question first**, then choose the model — not the other way around.
2. **Report your full pipeline**: imputation method, encoding strategy, split ratio, and random seed (reproducibility starts here).
3. **Always use a stratified split** (or stratified k-fold cross-validation) for imbalanced clinical outcomes.
4. **Never report accuracy alone** for imbalanced problems — always include Precision, Recall, F1, and ROC-AUC (and consider PR-AUC too).
5. **Compare against a simple baseline** (like Logistic Regression) — a complex model is only worth publishing if it meaningfully beats the simple one.
6. **Interpret your model**, don't just report numbers: feature importance, coefficients, or explainability tools (e.g. SHAP) build clinical trust.
7. **Be explicit about your dataset's limitations** (single-hospital data, small sample size, missing subgroups) in your Discussion section.
8. **Make your code and (de-identified) data available** wherever ethically and legally possible — this is how the field, and your citation count, grows.

## D.3 ⚠️ Ethics & Responsible AI Use in Public Health

- **Never upload real patient data to public, consumer AI/LLM tools** (ChatGPT, Claude, Gemini, or any similar service accessed through a regular consumer account) unless your institution has an approved, compliant data-processing agreement in place. Even "anonymized" data can sometimes be re-identified.
- Perform all real analysis **locally, or on your institution's approved, secured infrastructure** — exactly the workflow you practiced these two days, on your own machine / Colab runtime, using open datasets.
- Always obtain proper **ethics board (IRB) approval** before working with real patient records, and follow your institution's data governance policy strictly.
- Be transparent in publications about **which tools, models, and versions** were used (reproducibility again!).
- Remember that a model predicting "stroke risk" is a **decision-support tool**, not a diagnostic replacement for a clinician — always frame your work accordingly, and discuss potential biases (e.g. underrepresented age groups, regions, or genders in your training data).
- Consider the **downstream impact** of false negatives vs. false positives in your specific clinical context, and choose your decision threshold and metrics accordingly (we saw this directly in Part D.5 of Day 1!).

## D.4 Thank You! 🙏

Thank you all so much for your energy and curiosity throughout these two days. You now have hands-on experience across the **entire modern ML toolkit** — from `if`/`else` statements all the way to PyTorch neural networks — applied to a real, meaningful public-health problem.

We hope this notebook, and the dataset and code behind it, become a genuinely useful foundation for your thesis work and future research in AI for Public Health.

Go build something that helps people. 💙

— **Teerapong Panboonyuen (Kao)**
[kaopanboonyuen.github.io](https://kaopanboonyuen.github.io/)
